## Import Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Mount your google drive
from google.colab import drive
drive.mount('/content/drive')

.

.

.



## Load Feature Dataset & P-value Rank
* Results of DA3-1 and DA3-2

In [ ]:
FeatureData = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/SavedFiles/FeatureData.csv', sep=',', header=None)
FeatureData.shape

In [ ]:
P_value_Rank = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/SavedFiles/P_value_Rank.csv' , sep=',' , header=None)
P_value_Rank

## Select Target Features Using P-value

In [ ]:
# Select features from StartRank to StartRank+Num
StartRank = 1
Num       = 30

SelectedFeatues = np.zeros((Num, FeatureData.shape[1]))

s = 0
for i in range(StartRank, StartRank+Num):
    index                 = int(P_value_Rank.iloc[i-1,0])
    SelectedFeatues[s,:]  = FeatureData.iloc[index,:].values
    s += 1

FeatureSelected = pd.DataFrame(SelectedFeatues).T
FeatureSelected

.

.

.

## Data Standardization / Normalization (Min-Max scaling)

### Why scale features before PCA?
- PCA is variance-based: features with larger numeric ranges dominate the covariance matrix and can swamp others.  
- Scaling puts features on comparable scales so that PCA captures *structure*, not unit choices.


Packages for Principal Component Analysis (PCA)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler

#### Standardization of features

**Concept**
- Standardization (z-score) rescales each feature to mean `0` and standard deviation `1`:  [$z = \frac{x - \mu}{\sigma}$]

- Good default for PCA because it equalizes feature variances.

**What the code does**
- `StandardScaler().fit_transform(FeatureSelected)`:
  - `fit` learns per-feature mean and std from `FeatureSelected`.
  - `transform` returns standardized values.
- Wrap with `pd.DataFrame(...)` to keep a tidy tabular object.

**When to prefer**
- When features are roughly Gaussian-like or measured on different units/scales.

In [ ]:
FeatureSelected_std = StandardScaler().fit_transform(FeatureSelected)
FeatureSelected_std = pd.DataFrame(FeatureSelected_std)
FeatureSelected_std

#### Normalization of features (Min-Max)

**Concept**
- Min-Max scaling maps each feature into `[0, 1]`:
\[
$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$
\]

- Preserves relative distances and shapes but compresses outliers into the edges.

**What the code does**
- `MinMaxScaler().fit_transform(FeatureSelected)` scales each column to `[0,1]`.  
- Again, wrap with `pd.DataFrame(...)`.

**When to prefer**
- When the downstream model assumes bounded inputs (e.g., some neural nets) or you want all features strictly in `[0,1]`.

**Caution**
- Very sensitive to outliers because `x_min`/`x_max` are extreme values.

In [ ]:
FeatureSelected_norm = MinMaxScaler().fit_transform(FeatureSelected)
FeatureSelected_norm = pd.DataFrame(FeatureSelected_norm)
FeatureSelected_norm

## Visualizing the effect of scaling (boxplots)

**What the plots show**
- **Features (no scaling):** raw feature ranges vary widely; medians and interquartile ranges (IQRs) are not comparable.  
- **Standardized Features:** medians near `0`, IQR ~ `1`; spread is comparable across features.  
- **Normalized Features:** all values squeezed into `[0,1]`; relative ordering preserved.

**Why this matters**
- These diagnostics justify scaling choice before PCA and help spot outliers or skewness.

In [ ]:
Start = 0  # Number of feature starts with '0'
End   = 3

plt.figure(figsize=(15,4))
plt.subplot(1,3,1)
plt.boxplot(FeatureSelected.iloc[:,Start:End+1].T)
plt.title('Features (no scaling)', fontsize=15)
plt.xlabel('Features')

plt.subplot(1,3,2)
plt.boxplot(FeatureSelected_std.iloc[:,Start:End+1].T)
plt.title('Standardized Features', fontsize=15)
plt.xlabel('Features')

plt.subplot(1,3,3)
plt.boxplot(FeatureSelected_norm.iloc[:,Start:End+1].T)
plt.title('Normalized Features', fontsize=15)
plt.xlabel('Features')

plt.show()

.

.

.

## PCA Implementation

**Goal**
- Reduce dimensionality while retaining as much variance as possible.

**Key ideas**
- PCA finds orthogonal directions (principal components, PCs) that maximize variance.  
- Projecting data onto the first `k` PCs yields a `k`-dimensional representation.

**What the code does**
- Set `dim = 2` to compute two principal components.  
- `pca = PCA(n_components=dim)` initializes PCA.  
- `PC = pca.fit_transform(FeatureSelected_std)` fits on **standardized** features and returns the 2-D scores:
  - `PC[:,0]` → scores on PC1 (largest variance).
  - `PC[:,1]` → scores on PC2 (second largest, orthogonal to PC1).

**Why standardize before PCA**
- Without scaling, features with large units dominate the covariance matrix and distort component directions.

In [ ]:
# Extract Principal Components (PC)
dim = 2  # Dimension to reduce
pca = PCA(n_components = dim)
PC  = pca.fit_transform(FeatureSelected_std)
pd.DataFrame(PC)

## Visualization of PCA Result (2-D)

**What the scatter shows**
- Each point = one sample projected into the PC1–PC2 plane.  
- Coloring by class label (e.g., **Normal** vs **Abnormal**) helps assess separability in reduced space.

**Reading the plot**
- Clear clusters or margins suggest that a low-dimensional representation preserves class structure.  
- Overlap indicates you may need more components, different features, or non-linear methods.


In [ ]:
NoOfData = int(FeatureData.shape[1]/2)

plt.figure(figsize = (7,5))
plt.scatter(PC[:NoOfData,0], PC[:NoOfData,1], color='b', marker='o', label='Normal')
plt.scatter(PC[NoOfData:,0], PC[NoOfData:,1], color='r', marker='o', label='Abnormal')
plt.legend(fontsize=15)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(alpha=0.5)
plt.show()

.

.

.

.

# Task

- Visualize the bottom 30 features with the largest p-values in 2-D space through PCA.
  - Select the **30 features with the largest p-values** and visualize them in 2-D using PCA.  
  - Only modify the **feature selection number** in the code.  
  - No need to re-import packages.  
  - Compare the result with the previous visualization (using features with the smallest p-values).

In [ ]:
# Select Target Features for PCA




# Data Standardization (No Normalization)




# PCA Implementation




# Visualization of PCA Result





<details>
<summary>Click to see Answer </summary>

```python
# Select Target Features for PCA
StartRank = 240
Num       = 30

SelectedFeatues = np.zeros((Num, FeatureData.shape[1]))

s = 0
for i in range(StartRank, StartRank+Num):
    index                 = int(P_value_Rank.iloc[i-1,0])
    SelectedFeatues[s,:]  = FeatureData.iloc[index,:].values
    s += 1

FeatureSelected = pd.DataFrame(SelectedFeatues)
FeatureSelected = FeatureSelected.T


# Data Standardization (No Normalization)
FeatureSelected_std = StandardScaler().fit_transform(FeatureSelected)
FeatureSelected_std = pd.DataFrame(FeatureSelected_std)


# PCA Implementation
dim = 2  # Dimension to reduce
pca = PCA(n_components = dim)
PC  = pca.fit_transform(FeatureSelected_std)


# Visualization of PCA Result
NoOfData = int(FeatureData.shape[1]/2)

plt.figure(figsize = (7,7))
plt.scatter(PC[:NoOfData,0], PC[:NoOfData,1], color='b', marker='o', label='Normal')
plt.scatter(PC[NoOfData:,0], PC[NoOfData:,1], color='r', marker='o', label='Abnormal')
plt.legend(fontsize=15)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(alpha=0.5)
plt.show()